# LFM Instance Segmentation Example Workflow
This notebook is an example workflow of doing semantic segmentation on visible light, UV, and static bands of Lunar data. 

## Purpose of this notebook
This notebook is designed to be used as an example semantic segmentation workflow ("crater" vs "non crater" model prediction). A pretrained Graha model is loaded from disk, and we build our own crater detection model on top of this. The model loads data from the lfm project space, then runs several epochs of training on this data, and finally visualizes the model performance on the validation dataset. If you would like to control some of the model parameters, see the "User Configuration" section below.  

**Note**: currently, the training dataset used in this notebook is using 7-band WAC data. If you would like to filter out certain WAC/Static bands, you can filter them using the BAND_FILTER variable in the config. **See the README in the [LFM repo](https://github.com/nasa-nccs-hpda/lfm) for more info.** 

## Imports, Dino Repo Clone

In [1]:
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_OFFLINE'] = '0'

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

from functools import partialmethod
from tqdm import tqdm
tqdm.__init__ = partialmethod(tqdm.__init__, disable=False)

import sys
import torch
import subprocess
import warnings
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from lightning.pytorch import seed_everything

In [5]:
# Get the repo root directory (parent of notebooks/)
repo_root = Path.cwd().parent.parent

# Convert /panfs path to /explore path (JupyterHub quirk)
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)

# Verify we're in the right location
if not (repo_root / "lfm").exists():
    raise FileNotFoundError(
        "Cannot find lfm/ directory. "
        "Please ensure you're running this notebook from the lfm/notebooks/ directory."
    )

# Add the parent of the repo to sys.path for imports
sys.path.insert(0, str(repo_root))

# Import required modules
from lfm.all_models.all_tasks import SingleModelExperiment
from lfm.all_models.inst_seg.testing.instance_test_suite_callback import (
    GrahaInstancePlotCallback,
    InstanceEpochTestSuiteCallback,
)
from lfm.all_models.all_tasks.utils import ensure_data_symlink
from scripts.python.instance_seg import instance_seg_comparison as comparison
from lfm.all_models.all_tasks.utils import (
    plot_instance_cache_predictions,
    save_graha_instance_prediction_cache,
)
from lfm.full_model.inst_seg import instance_graha_components
deps = instance_graha_components.import_project_dependencies()
print("✓ Successfully imported LFM modules")

✓ Successfully imported LFM modules


## User Configuration

#### Paths
`DATA_ROOT`: there are multiple different dataset configurations under the lfm/model_inputs/300_300_inputs folder. Choose any of the band subdirectories in the 300_300_inputs folder, and leave the other variables under "data paths" (IMAGE_DIR, LABEL_DIR, etc) as they are. 

`OUTPUT_DIR`: this is a relative path, so by default the outputs (visualizations, model checkpoints, dataset statistics) will go to the same folder as the notebook. 

#### Dataset parameters
`MAX_SAMPLES`: number of training samples to look for in DATA_ROOT. 

#### Training hyperparameters
`BATCH_SIZE`: best to leave this at 16 to conserve VRAM, especially at higher number of input bands (>7). 

`NUM_EPOCHS`: default value is 1 for demonstration purposes, the model tends to start to get its best results around 100 epochs. Feel free to experiment with this.

`BASE_LR`,`WEIGHT_DECAY`: feel free to change these to adjust how aggressively the model tries to tune to each training batch. Higher means more aggressive model learning, but can also mean the model has a harder time converging to the correct result. 

#### Model hyperparameters
`FREEZE_ENCODER`: whether to keep the Dino backbone frozen; we found better results with False, but feel free to try with freezing set to true. 

`NUM_BANDS`: number of bands to include in the input. Currently supported are 3/5/7/12-band inputs.

`BAND_FILTER`: list of band indices in the range [0, 11], 0-indexed, to use in the dataset. For example, if I want to use only VIS bands, I would supply [0, 1, 2, 3, 4]. Band ordering is the following for the 12-band dataset: VIS (0-4), UV (5, 6), KAGUYA STATIC (7-11). 

- <mark>Note: the toy model expects a minimum of 3 bands (RGB). Ensure the band filter has at least 3 vis bands, ideally vis bands [3, 1, 0] which are closest in wavelength to RGB. </mark>

In [ ]:
BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "instance_seg_finetuning"
print(f"Output directory: {BASE_OUTPUT_DIR}")

PRETRAIN_DIR = "/explore/nobackup/projects/lfm/ibm_model_pretrain_dir"
# GRAHA_INPUT_MODALITY_MODE = "vis-uv"  # Use "vis-uv" to reuse pretrained 5-band vis + 2-band uv modalities.
# GRAHA_VIS_UV_MERGE_METHOD = "mean"
LIGHTNING_CHECKPOINT = None  # ?

# CROP_SIZE = 256
# STATS_BATCH_SIZE = 16
BATCH_SIZE = 8
NUM_WORKERS = 10
MAX_EPOCHS = 1

BACKBONE_LR = 5.0e-5
HEAD_LR = 2.0e-4
LAYER_DECAY = 0.75
WEIGHT_DECAY = 0.05
WARMUP_STEPS = 500

BAND_FILTER = [0, 1, 2, 3, 4, 5, 6]
# ANCHOR_SIZES = [[8], [16], [32], [64]]
# ANCHOR_ASPECT_RATIOS = [0.5, 1.0, 2.0]
# SCORE_THRESHOLD = 0.5
# PLOT_PREDICTIONS = True
# PREDICTION_SPLIT = "val"
# PREDICTION_N_SAMPLES = 5
# PREDICTION_SCORE_THRESHOLD = 0.5
# MASK_SHIFT = (0, 0)  # (x_pixels, y_pixels): positive moves labels right/down
# SEED = 42

# RUN_FIT = False
# LOSS_SMOKE_ONLY = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Data path
DATA_ROOT = "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/inst_seg"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs/inst_seg"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Dataset parameters
MAX_SAMPLES = 500  # Set to None to use all samples, or an integer to limitn

# Training hyperparameters
BATCH_SIZE = 8  # Number of images fed into the model at a time
NUM_EPOCHS = 1 # 10 is the default, increase for more model learning
BASE_LR = 5e-5  # Starting learning rate
WEIGHT_DECAY = 1e-3  # Weight decay of optimizer

# Model parameters
FREEZE_ENCODER = False  # Whether to keep the model weights frozen
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  # Bands to keep, in order of filtering

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
args = Namespace(
    data_root=DATA_ROOT,
    base_output_dir=str(BASE_OUTPUT_DIR) if BASE_OUTPUT_DIR is not None else None,
    pretrain_dir=str(PRETRAIN_DIR) if PRETRAIN_DIR is not None else None,
    graha_input_modality_mode=GRAHA_INPUT_MODALITY_MODE,
    graha_vis_uv_merge_method=GRAHA_VIS_UV_MERGE_METHOD,
    lightning_checkpoint=str(LIGHTNING_CHECKPOINT) if LIGHTNING_CHECKPOINT is not None else None,
    crop_size=CROP_SIZE,
    stats_batch_size=STATS_BATCH_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    max_epochs=MAX_EPOCHS,
    backbone_lr=BACKBONE_LR,
    head_lr=HEAD_LR,
    layer_decay=LAYER_DECAY,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    anchor_sizes=ANCHOR_SIZES,
    anchor_aspect_ratios=ANCHOR_ASPECT_RATIOS,
    score_threshold=SCORE_THRESHOLD,
    plot_predictions=PLOT_PREDICTIONS,
    prediction_split=PREDICTION_SPLIT,
    prediction_n_samples=PREDICTION_N_SAMPLES,
    prediction_score_threshold=PREDICTION_SCORE_THRESHOLD,
    mask_shift=MASK_SHIFT,
    seed=42,
    no_fit=not RUN_FIT,
    loss_smoke_only=LOSS_SMOKE_ONLY,
)

seed_everything(42)
config = workflow.build_config(args)
workflow.configure_python_paths(config)
workflow.print_config(config)
workflow.validate_required_paths(config)
deps = workflow.import_project_dependencies()

## Create output directory
This will contain model checkpoints and visualizations. It will ask you to overwrite the directory if it already exists, so ensure you copy files that you want to keep to a different directory!

In [ ]:
OUTPUT_DIR = create_timestamped_output_dir(OUTPUT_DIR)

## Create datamodule
1. Load pretraining stats from .yaml file
2. Create datamodule using pretraining stats and other config options

In [ ]:
# STEP 1: pretraining stats
print("\nSTEP 1: Loading pretraining stats...")
print("="*60)

datamodule_cls = deps["GrahaObjectDetectionInstanceDataModule"]
means, stds = instance_graha_components.get_normalization_stats(
    graha_config,
    datamodule_cls,
)

print("Done.")

In [ ]:
print("\nSTEP 2: Creating datamodule...")
print("="*60)

graha_datamodule = instance_graha_components.create_datamodule(
    graha_config,
    datamodule_cls,
    means,
    stds,
)

print("Done.")

## Create Terratorch Task Object, Model

In [ ]:
task_cls = instance_graha_components.make_downstream_object_detection_task_class(
    deps["LunarObjectDetectionTask"]
)

graha_task = instance_graha_components.create_task(
    graha_config,
    task_cls,
    graha_sample_batch,
)

In [ ]:
print("\n" + "="*60)
print("Loading mask2former Dino model...")
print("="*60)

model = _()

## Run Training

In [ ]:
# create trainer
trainer = None

In [ ]:
print("\n" + "="*60)
print("Starting training.")
print("="*60)

trainer.fit()

## Display some of the output visualizations

The training of the model is already producing some visualizations every N epochs.
Here we open some of the visualizations to look at them from the notebook.

In [ ]:
visualization_dir = os.path.join(OUTPUT_DIR, 'visualizations')
visualization_filenames = sorted(glob(os.path.join(visualization_dir, '*.png')))

In [ ]:
for vis_filename in visualization_filenames:
    img = mpimg.imread(vis_filename)
    plt.figure(figsize=(16, 14))
    plt.imshow(img)
    plt.show()

In [ ]:
if 'model' in globals():
    del model
torch.cuda.empty_cache()